# Wall design — complete tour

Covers every constructor argument, every output, every design mode,
manual overrides, unit hot-swap and the iteration loop.

Pipeline:
1. Materials + bar schedule
2. Web rebar layout (no BE yet)
3. WallSection construction
4. Wall construction — every argument
5. Lazy run + scalars
6. PM curves (in-plane / out-of-plane)
7. PMM surface — volume + slice
8. wall.check(demands)
9. wall.design() — three modes + envelope
10. Iteration history
11. Original vs final wall
12. BE as a Column — design, optimize, plot
13. evolve() manually
14. Unit hot-swap
15. report() and summary()

## 0. Path setup

This notebook lives inside `design/`. We add the repo root to
`sys.path` so `from design import ...` resolves regardless of
where Jupyter was launched.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'design' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('repo root:', repo_root)

In [ ]:
from design import (
    Concrete, Steel, Bar, WallSection, Wall,
    BarSchedule, PlotStyle, to_internal, units_from,
)
from design.walls import WallDemands
from design.sections.reinforcement import RebarLayout, perimeter_bars

INPUT_UNITS   = 5    # kN_mm_C — what we type below
DISPLAY_UNITS = 11   # Tonf_mm_C — plots / reports / summaries

## 1. Materials and bar schedule

In [ ]:
concrete = Concrete(fc=35, unit_weight=24)
steel    = Steel(fy=420, grade=60, density=7850)

schedule = BarSchedule(
    longitudinal=[12, 16, 20, 22, 25, 28, 32],
    hoops=[10, 12, 16],
)
concrete, steel, schedule

## 2. Web rebar layout

Single perimeter group along the full wall length. No boundary elements yet — we let `wall.design()` propose them.

In [ ]:
web = perimeter_bars(
    b=300, h=4000, cover=50, n_x=2, n_y=14,
    bar=Bar(diameter=12), steel=steel,
)
print('web bars:', len(web.bars))
print('web As  :', web.total_area, 'mm^2')

## 3. WallSection

In [ ]:
section = WallSection(
    lw=4000, tw=300,
    concrete=concrete,
    rebar=RebarLayout(groups=(web,)),
)
print(section)
print('Ag   =', section.gross_area(), 'mm^2')
print('Acv  =', section.Acv, 'mm^2')
print('has BE?', section.has_boundary_elements)

## 4. Wall — every argument

In [ ]:
wall = Wall(
    section=section,
    hw=30_000,                  # total wall height (mm)
    lu=3_000,                   # unbraced storey height (mm)
    rho_t=0.0025,               # initial distributed transverse ratio
    rho_l=0.0025,
    fyt=420.0,                  # MPa
    seismic=True,
    omega_v_factor=1.5,         # §18.10.3.1.2
    hx=200.0,                   # §18.7.5.2 inside the BE
    transverse_bar_diameter=10.0,
    web_bar_diameter=12.0,
    units=DISPLAY_UNITS,
    bar_schedule=schedule,
    label='W-1A',
)
wall

### Equivalente con `Wall.rectangular()`

Las celdas anteriores construyen `web → section → wall` paso a paso. Para uso típico, el factory `Wall.rectangular()` produce el mismo objeto en una sola llamada — paridad con `Beam.rectangular()` y `Column.rectangular()`.

In [ ]:
# Wall.rectangular() — factory parity con Beam/Column. Acá vemos
# un muro BARBELL: los boundary elements son más anchos que el
# alma (be_top_thickness = 500 mm > tw = 300 mm) y llevan
# armado column-style (n_x_be = 4 barras de ø22 a través del
# espesor del BE, 5 a lo largo de su longitud).
#
# ACI 318-25 §18.10.6.4(b): los BE de muros especiales deben
# detallarse como columnas — n_x_be < 3 lanza ValueError.
wall_barbell = Wall.rectangular(
    lw=4000, tw=300, hw=30_000, lu=3_000,
    concrete=concrete, steel=steel,
    # web
    n_y_web=14, db_web=12, cover=50,
    # BARBELL boundary elements
    be_top_length=600, be_top_thickness=500,
    be_bot_length=600, be_bot_thickness=500,
    n_x_be=4, n_y_be=5, db_be=22,
    # transverse / seismic
    db_stirrup=10, hx=200,
    seismic=True, omega_v_factor=1.5,
    units=DISPLAY_UNITS,
    bar_schedule=schedule,
    label='W-1-barbell',
)
print('barbell built:', wall_barbell)
print()
print(f'  tw (web)         = {wall_barbell.section.tw:.0f} mm')
print(f'  BE thickness     = {wall_barbell.section.be_thickness_top:.0f} mm  '
      f'(barbell: > tw)')
print(f'  BE length        = {wall_barbell.section.be_length_top:.0f} mm')
print(f'  n_x_be (column-style) = {wall_barbell.n_x_be}')
for i, g in enumerate(wall_barbell.section.rebar.groups):
    tag = ['web', 'BE top', 'BE bot'][i] if i < 3 else f'group {i}'
    print(f'  {tag:8s}: {len(g.bars):2d} bars, As = {g.total_area:>5.0f} mm^2')

### Muros asimétricos (tipo C / T / Z)

Cuando el BE no puede centrarse sobre el eje del alma — porque el muro choca con un núcleo de ascensor, una escalera, una losa que continúa solo de un lado, etc. — se desplaza en la dirección del espesor para que UNA cara del BE coincida con una cara del alma.

El factory acepta `be_align='left'|'center'|'right'` (default `'center'` = barbell simétrico):

- `'left'`  → el BE protruye solo hacia la **izquierda** (cara derecha del BE = cara derecha del alma).
- `'right'` → el BE protruye solo hacia la **derecha** (cara izquierda del BE = cara izquierda del alma).

Se puede mezclar por extremo con `be_top_align` / `be_bot_align` para casos tipo Z (BE top a un lado, BE bot al otro).

In [ ]:
# Tipo C: AMBOS BE protruyen hacia la izquierda.
# La cara DERECHA del BE coincide con la cara derecha del alma,
# así el muro queda 'cerrado' por ese lado (típico cuando ahí
# hay otra estructura — losa, núcleo, escalera).
wall_C = Wall.rectangular(
    lw=4000, tw=300, hw=30_000, lu=3_000,
    concrete=concrete, steel=steel,
    # web
    n_y_web=14, db_web=12, cover=50,
    # BARBELL geometry — BE más anchos que el alma
    be_top_length=600, be_top_thickness=500,
    be_bot_length=600, be_bot_thickness=500,
    n_x_be=4, n_y_be=5, db_be=22,
    # ASIMETRIA: ambos BE flush con el lado derecho del alma
    be_align='left',
    db_stirrup=10, hx=200,
    seismic=True, omega_v_factor=1.5,
    units=DISPLAY_UNITS,
    bar_schedule=schedule,
    label='W-1-C',
)
print('C-shape built:', wall_C)
print()
bb = wall_C.section.bounding_box()
print(f'  tw (web)              = {wall_C.section.tw:.0f} mm')
print(f'  BE thickness          = {wall_C.section.be_thickness_top:.0f} mm')
print(f'  BE x-offset top/bot   = '
      f'{wall_C.section.be_top_x_offset:+.0f} / '
      f'{wall_C.section.be_bot_x_offset:+.0f} mm')
print(f'  bbox x-range          = [{bb[0]:+.0f}, {bb[2]:+.0f}] mm '
      f'(asimetrico)')
print(f'  web ocupa             = [{-wall_C.section.tw/2:+.0f}, '
      f'{+wall_C.section.tw/2:+.0f}] mm')
print(f'  BE ocupa              = '
      f'[{wall_C.section.be_top_x_offset - wall_C.section.be_thickness_top/2:+.0f}, '
      f'{wall_C.section.be_top_x_offset + wall_C.section.be_thickness_top/2:+.0f}] mm '
      f'(comparte la cara x={wall_C.section.tw/2:+.0f} con el alma)')
print()
# Plot del C-shape para ver la geometria
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(3, 8))
wall_C.plot.section(ax=ax)
ax.set_title('Muro tipo C')
plt.show()

## 5. Lazy run + scalars

In [ ]:
wall.run(n_angles=19, n_points=60)
print(f'Po         = {wall.Po:.0f} kN')
print(f'To         = {wall.To:.0f} kN')
print(f'phi_Pn_max = {wall.phi_Pn_max:.0f} kN')
print(f'phi_Vn     = {wall.phi_Vn:.0f} kN  (cap = {wall.Vn_max:.0f} kN)')

## 6. PM curves (in-plane / out-of-plane)

In [ ]:
diagram_in  = wall.surface[0.0]
diagram_out = wall.surface[90.0]
print(f'in-plane:     max phi_Mn = {max(diagram_in.phi_Mn):.0f} kN.m, Po = {diagram_in.Po:.0f} kN')
print(f'out-of-plane: max phi_Mn = {max(diagram_out.phi_Mn):.0f} kN.m')

## 7. PMM surface — volume + slice

In [ ]:
Pu_grid, Mx_grid, My_grid = wall.surface.volume(n_Pu=20)
print('grid shape:', Mx_grid.shape, '(Pu_slices, theta_points)')

mx, my = wall.surface.at_Pu(2000.0)
print(f'@ Pu=2000:  Mx in [{mx.min():.0f}, {mx.max():.0f}], My in [{my.min():.0f}, {my.max():.0f}]')

ok, ratio = wall.surface.check(Pu=2000, Mux=8000, Muy=200)
print(f'biaxial check: ok = {ok}, ratio = {ratio:.3f}')

## 8. wall.check(demands)

In [ ]:
demands = WallDemands(
    Pu=2000.0, Mu=8000.0, Mu_out=200.0,
    Vu=1500.0,
    delta_u=180.0,
    sigma_max=8.0,
)
chk = wall.check(demands)
print(f'ratio PM        = {chk.ratio_pm:.3f}')
print(f'ratio shear     = {chk.ratio_shear:.3f}')
print(f'BE disp/stress  = {chk.be_required_disp} / {chk.be_required_stress}')
print(f'BE length req   = {chk.be_length_required:.0f} mm')
print(f'BE thickness min= {chk.be_thickness_min:.0f} mm')
print(f'distrib rho ok? {chk.distributed_rho_ok}')
print(f'c at demand     = {chk.c_at_demand:.0f} mm')
print(f'PASSED          = {chk.passed}')

## 9. wall.design() — three modes + envelope, with auto_update

In [ ]:
results = wall.design(demands, auto_update=True, max_iter=5, tolerance=25.0)
print(f'iterations : {results.iterations}')
print(f'converged  : {results.converged}')
print()
print('--- ENVELOPE ---')
env = results.envelope
print(f'mode               : {env.mode}')
print(f'rho_t required    : {env.rho_t_required:.4f}')
print(f'rho_l required    : {env.rho_l_required:.4f}')
print(f'double curtain?    : {env.double_curtain_required}')
print(f'BE required        : {env.be_required}')
print(f'BE length proposed : {env.be_length_proposed:.0f} mm')
print(f'BE thick proposed  : {env.be_thickness_proposed:.0f} mm')
print(f'Mpr in-plane       : {env.Mpr_in_plane:.0f} kN.m')
print(f'omega_v            : {env.omega_v}')
print(f'Ve capacity        : {env.Ve_capacity:.0f} kN')
print(f'phi_Vn (proposed)  : {env.phi_Vn:.0f} kN')

## 10. Iteration history

In [ ]:
for i, w in enumerate(results.history):
    print(f'  iter {i}: BE top = {w.section.be_length_top:>4.0f} mm, BE bot = {w.section.be_length_bot:>4.0f} mm')

## 11. Original vs final wall

In [ ]:
print('ORIGINAL:', results.original_wall)
print('FINAL   :', results.final_wall)
print()
print('Original section:')
results.original_wall.summary()
print()
print('Final section:')
results.final_wall.summary()

## 12. BE as a Column — full columns/ API works

We can design / optimize / inspect each boundary element exactly
like a stand-alone SMF column.

In [ ]:
final = results.final_wall
if final.has_boundary_elements:
    be = final.be_top
    print(be)
    print()
    be_results = be.design()
    envb = be_results.envelope
    print(f'BE detailing (envelope):')
    print(f'  Ash_x req = {envb.ash_required_x:.0f} mm^2  provided {envb.av_provided_x:.0f} ({envb.n_legs_x} legs)')
    print(f'  Ash_y req = {envb.ash_required_y:.0f} mm^2  provided {envb.av_provided_y:.0f} ({envb.n_legs_y} legs)')
    print(f'  db_hoop   = {envb.db_hoop:.0f} mm')
    print(f'  spacing   = {envb.spacing_confined:.0f} mm (lo)')
    print(f'  hx ok?    = {envb.hx_ok}')
else:
    print('No BE — demands did not trigger §18.10.6.')

## 13. evolve() manually

In [ ]:
# Suppose we want to explicitly grow the BE to 800 mm
manual_wall = results.final_wall._with_section(
    results.final_wall.section.with_boundary_elements(
        top_length=800.0, bot_length=800.0,
    )
)
print(manual_wall)
manual_wall.run()
print(f'Po = {manual_wall.Po:.0f} kN, phi_Vn = {manual_wall.phi_Vn:.0f} kN')

## 14. Unit hot-swap

In [ ]:
for code in (5, 11, 6, 12):    # kN_mm_C, Tonf_mm_C, kN_m_C, Tonf_m_C
    results.final_wall.set_units(code)
    u = results.final_wall.units
    print(f'{u.name:>11s}: Po = {results.final_wall.Po*u.force_factor:>8.0f} {u.force}')

## 14b. Reinforcement quantities — optimize sweep

`wall.run_optimize(demands)` is the wall analogue of
`beam.run_optimize()` and `column.run_optimize()`. It
enumerates feasible combinations of

  * web distributed bars `(db_web, spacing, layers)`,
  * BE longitudinal bars `(db_be, n_y_be)` when the wall
    has boundary elements,

computes the steel mass in kg/m³ of concrete and ranks the
alternatives ascending by `rho_total`. The baseline
(detailing that `wall.design()` would have used) is marked
with the `baseline` tag.

Three things this sweep does NOT vary:

  * BE geometry (length, thickness, x-offset) — fixed by
    construction. To explore those, build several walls.
  * BE confinement (db_hoop, n_legs, spacing) — that lives
    in `wall.be_top.run_optimize()` (the BE is a Column).
  * n_x_be (bars across the BE thickness) — fixed at
    construction; default 3 per §18.10.6.4(b).

In [ ]:
from design.walls.optimize import format_optimize_table

# Run the sweep on the FINAL wall (after design()'s
# auto_update has added the BE). Pass the same demands.
alts = results.final_wall.run_optimize(demands)
print(f'Alternatives explored : {len(alts)}')
print(f'Lightest rho_total    : {alts[0].rho_total:.1f} kg/m^3')
print(f'   detail: {alts[0].detailing}')
print()
print(format_optimize_table(alts, top_n=12))

**Reading the table**

  * `rho_t` is the web horizontal (shear) reinforcement.
  * `rho_l` lumps the web vertical bars and the BE
    longitudinal bars (both contribute to flexural
    capacity).
  * `rho_tot = rho_t + rho_l`.
  * The detail string reads `web phi<db>@<s>x<layers>c |
    BE <n_y_be>xphi<db_be>`. For walls without BEs the BE
    half disappears automatically.

If the lightest row is much lighter than the baseline,
consider applying it with `wall.evolve(alts[0].proposal)`
to swap in the cheaper detailing — then `check(demands)`
again to verify the demand still clears it.

## 15. Report + plots

In [ ]:
results.final_wall.set_units(DISPLAY_UNITS)
import json
print(json.dumps(results.final_wall.report(), indent=2, default=str))

In [ ]:
import matplotlib.pyplot as plt
results.final_wall.plot.dashboard(demands=demands)
plt.show()

In [ ]:
results.final_wall.plot.iteration_history(results)
plt.show()

In [ ]:
if results.final_wall.has_boundary_elements:
    results.final_wall.be_top.plot.section()
    plt.show()